# Notebook 4 · Intermediate LangChain

### Memory, structured output, and middleware that guards the loop

A single agent turn is a demo. A product remembers the last thing you said, returns data your code can trust, hides private information, and stops to ask before it does something expensive. All four are one concept away from what you already know, and all four run offline below.

**Standalone setup.** Run the cell first.

```bash
pip install langchain langgraph langchain-aws pydantic
```

In [ ]:
from typing import Any, List
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class ScriptedChatModel(BaseChatModel):
    '''Deterministic, credential-free stand-in. Framework code is real; replies are scripted.'''
    responses: List[Any]
    idx: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self.responses[min(self.idx, len(self.responses) - 1)]
        object.__setattr__(self, "idx", self.idx + 1)
        return ChatResult(generations=[ChatGeneration(message=reply)])

    def bind_tools(self, tools, **kwargs):
        return self


def show_trace(result):
    for m in result["messages"]:
        label = type(m).__name__
        if getattr(m, "tool_calls", None):
            c = m.tool_calls[0]
            print(f"{label:14} -> calls {c['name']}({c['args']})")
        else:
            print(f"{label:14} -> {m.content}")


print("setup ready")

setup ready


---
## 1. Memory: remember across turns

By default an agent is amnesiac. Each `invoke` starts fresh. To hold a conversation you give it a checkpointer and a `thread_id`. The checkpointer saves the message list after each turn, keyed by thread. Next turn on the same thread, the whole history comes back automatically.

```mermaid
flowchart LR
    T1[turn 1] --> CP[checkpointer saves state under thread_id]
    CP --> T2[turn 2 loads state model sees turn 1]
```

> **What runs next** one agent, two turns on the same thread. The second turn answers using what the first turn established.
> **Python construct** a config dict carrying `thread_id`, passed on every call.
> **LLM concept** the model has no built-in memory. Continuity is the checkpointer feeding prior messages back into context each turn.

In [2]:
model = ScriptedChatModel(responses=[
    AIMessage(content="Noted: PNR JX48Q2, Gold tier."),
    AIMessage(content="You are Gold tier, so the rebooking fee is waived."),
])

agent = create_agent(model, tools=[], checkpointer=InMemorySaver(), system_prompt="You are TravelMind.")
thread = {"configurable": {"thread_id": "rao-session"}}

first = agent.invoke({"messages": [{"role": "user", "content": "I'm Gold tier, my PNR is JX48Q2."}]}, thread)
print("after turn 1, messages in state:", len(first["messages"]))

second = agent.invoke({"messages": [{"role": "user", "content": "Do I owe a rebooking fee?"}]}, thread)
print("after turn 2, messages in state:", len(second["messages"]))
print("reply:", second["messages"][-1].content)

after turn 1, messages in state: 2
after turn 2, messages in state: 4
reply: You are Gold tier, so the rebooking fee is waived.


> **What just happened** turn two started with four messages already in state, not zero. The tier fact from turn one was in context when the model answered the fee question. Nothing was re-sent by the user. The checkpointer carried it.

> **Gotcha** the `thread_id` is the identity of the conversation. Reuse one id across two different users and they read each other's history. Generate a fresh id per user session, and never let it come from untrusted input.

---
## 2. Structured output: results your code can trust

Free text is fine for a human reader and useless for a program. If the next step is code (charge a fee, update a record, branch on a value), you need typed, validated fields, not a paragraph to parse with string tricks.

The idea: define the shape with Pydantic, then parse and validate against it. Validation is the safety net. A missing or wrong-typed field fails loudly instead of slipping through.

> **What runs next** define a schema, parse a good payload into typed fields, then watch a bad payload get rejected.
> **Python construct** a Pydantic `BaseModel`, `model_validate_json`, catching `ValidationError`.
> **LLM concept** models emit text. Turning that text into a trusted object is a separate, checkable step.

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Disruption(BaseModel):
    pnr: str = Field(description="the passenger name record")
    status: str = Field(description="cancelled, delayed, or on-time")
    rebook_fee_waived: bool = Field(description="true if the passenger is Gold tier")


good = '{"pnr":"JX48Q2","status":"cancelled","rebook_fee_waived":true}'
obj = Disruption.model_validate_json(good)
print("typed object:", obj)
print("branch on a real bool:", "no fee" if obj.rebook_fee_waived else "fee applies")

bad = '{"pnr":"JX48Q2","status":"cancelled"}'   # missing a required field
try:
    Disruption.model_validate_json(bad)
except ValidationError as e:
    err = e.errors()[0]
    print("\nrejected:", err["loc"], "->", err["msg"])

typed object: pnr='JX48Q2' status='cancelled' rebook_fee_waived=True
branch on a real bool: no fee

rejected: ('rebook_fee_waived',) -> Field required


> **What just happened** the good payload became an object with a real boolean you can branch on. The bad payload, missing `rebook_fee_waived`, was caught before it could corrupt anything downstream. That is the entire value: fail at the boundary, not three functions deep.

In production you do not hand-parse. You tell the agent the schema and it returns the validated object for you:

```python
# Reference (real model): the agent returns result["structured_response"] as a Disruption
agent = create_agent(model, tools=[...], response_format=Disruption)
result = agent.invoke({"messages": [{"role": "user", "content": "status of JX48Q2?"}]})
disruption = result["structured_response"]   # a validated Disruption instance
```

> **Skeptic's corner** why not just ask the model to "reply in JSON"? Because "usually valid JSON" is a bug generator. The schema plus validation is what turns "usually" into "guaranteed, or a clear error". Never trust unvalidated model output in code.

---
## 3. Middleware: hooks around the loop

Middleware runs your logic at fixed points in the loop without you rewriting the loop. Two hook points cover most needs: before the model sees the messages, and after the model responds but before tools run.

```mermaid
flowchart LR
    IN[user input] --> BM[before model: redact PII, summarise history]
    BM --> MODEL[model]
    MODEL --> AM[after model: pause for human approval]
    AM --> TOOLS[tools]
    TOOLS --> MODEL
```

Three built-in pieces earn their place immediately:

| Middleware | Hook | What it does |
|------------|------|--------------|
| `PIIMiddleware` | before model | redact or block private data before it reaches the model |
| `SummarizationMiddleware` | before model | compress old turns when the history gets long |
| `HumanInTheLoopMiddleware` | after model | stop and require approval before a risky tool runs |

### 3a. Redact private data before the model sees it

Card numbers, emails, and IPs should not flow into a model or a log if you can avoid it. `PIIMiddleware` catches them on the way in and replaces them.

> **What runs next** attach `PIIMiddleware` for credit cards, then send a message containing one, and inspect the stored message.
> **Python construct** passing a list of middleware to `create_agent`.
> **LLM concept** the model only ever sees the redacted text. The sensitive value never enters the prompt.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

model = ScriptedChatModel(responses=[AIMessage(content="Card saved to your profile.")])
agent = create_agent(
    model,
    tools=[],
    middleware=[PIIMiddleware("credit_card", strategy="redact", apply_to_input=True)],
    system_prompt="You are TravelMind billing.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "Save my card 4111 1111 1111 1111 for future rebookings"}]})
for m in result["messages"]:
    print(type(m).__name__, "->", m.content)

HumanMessage -> Save my card [REDACTED_CREDIT_CARD] for future rebookings
AIMessage -> Card saved to your profile.


> **What just happened** the human message in state now reads `[REDACTED_CREDIT_CARD]`. The raw number never reached the model. Swap `strategy="redact"` for `"block"` to refuse the turn outright, or `"mask"` to keep the last four digits. Built-in detectors cover email, credit_card, ip, mac_address, and url, and you can register your own.

> **Gotcha** redaction protects the model and your logs, not your tools. If a tool genuinely needs the real value, set `apply_to_tool_results` and `apply_to_input` deliberately, and keep the raw value out of anything you persist.

### 3b. Human approval before a risky action

Some actions should never fire on the model's say-so alone. Rebooking a flight, issuing a refund, sending an email: these want a human yes. `HumanInTheLoopMiddleware` interrupts the loop before the named tool runs and hands control back to you.

```mermaid
flowchart TD
    M[model: call rebook] --> G{approval gate}
    G -->|interrupt| H[human reviews the exact args]
    H -->|approve| RUN[tool runs]
    H -->|reject| SKIP[tool skipped, model told]
```

> **What runs next** gate the `rebook` tool. Invoke, catch the interrupt, show the pending action, then approve and let it finish. Requires a checkpointer so the paused state can be saved and resumed.
> **Python construct** reading `result["__interrupt__"]`, resuming with a `Command`.
> **LLM concept** the model proposes, the human disposes. The agent cannot take the irreversible step by itself.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware


@tool
def rebook(pnr: str, flight: str) -> str:
    '''Rebook a PNR onto a new flight. May charge a fare difference.'''
    return f"{pnr} rebooked onto {flight}, confirmation RBK-77"


model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "rebook", "args": {"pnr": "JX48Q2", "flight": "AI-506"}, "id": "r1", "type": "tool_call"}]),
    AIMessage(content="Done. JX48Q2 is confirmed on AI-506."),
])

agent = create_agent(
    model,
    tools=[rebook],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"rebook": True})],
    checkpointer=InMemorySaver(),
    system_prompt="You are TravelMind.",
)
thread = {"configurable": {"thread_id": "approve-1"}}

paused = agent.invoke({"messages": [{"role": "user", "content": "Rebook JX48Q2 onto AI-506"}]}, thread)
pending = paused["__interrupt__"][0].value["action_requests"][0]
print("PAUSED. awaiting approval for:", pending["name"], pending["args"])

# a human says yes
resumed = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), thread)
print("after approval:", resumed["messages"][-1].content)
print("tool actually ran:", any(type(m).__name__ == "ToolMessage" for m in resumed["messages"]))

PAUSED. awaiting approval for: rebook {'pnr': 'JX48Q2', 'flight': 'AI-506'}
after approval: Done. JX48Q2 is confirmed on AI-506.
tool actually ran: True


> **What just happened** the first invoke did not rebook anything. It stopped at the gate and returned the exact tool and arguments waiting for approval. Only after the `approve` decision did the tool run. The allowed decisions are `approve`, `reject`, `edit` (change the args first), and `respond` (reply without running). Rejecting skips the tool and tells the model, so it can explain to the user instead.

> **Gotcha** the gate is only as safe as the checkpointer. Without one, there is nowhere to save the paused state, and resume cannot work. Approval flows and persistence ship together.

### 3c. Summarisation: keep long chats inside the budget

Notebook 1 showed context growing until it overflows. `SummarizationMiddleware` handles it: when the history crosses a trigger, it compresses the older turns into a summary and keeps the recent ones intact, so the running total stays bounded.

```mermaid
flowchart LR
    LONG[long history over the trigger] --> SUM[summarise old turns]
    SUM --> KEEP[summary + recent turns]
    KEEP --> MODEL[model reads a bounded context]
```

```python
# Reference (real model): summarise once the history passes 40 messages, keep the last 20.
from langchain.agents.middleware import SummarizationMiddleware
agent = create_agent(
    model,
    tools=[...],
    middleware=[SummarizationMiddleware(model=model, trigger=("messages", 40), keep=("messages", 20))],
)
```

The `trigger` decides when to compress (by message count, token count, or a fraction of the window). `keep` decides how much recent detail survives verbatim. This is context engineering made operational: you are choosing what the agent is allowed to forget.

> **Skeptic's corner** summarisation is lossy by definition. If a detail from turn two matters at turn fifty, a summary may drop it. For facts that must persist, write them to memory or a store, do not rely on a summary to remember them for you.

---
## What you can now do

- Hold a multi-turn conversation with a checkpointer and a per-session thread id.
- Return typed, validated results with Pydantic, and reject bad payloads at the boundary.
- Redact private data before it reaches the model.
- Gate risky tools behind human approval, with pause and resume.
- Keep long conversations bounded with summarisation, and know what it costs you.

**Next, Notebook 5.** We drop below `create_agent` to the graph itself: custom nodes, deterministic routing you can unit test, and splitting one overloaded agent into several that coordinate, as a supervisor or as a swarm.

> **Skeptic's corner to carry forward** each piece here is a control you add on purpose, not a default you leave on. Memory, redaction, and approval have real costs in latency and complexity. Add them where the risk justifies them, not everywhere.